In [0]:
# 01 - BRONZE BATCH INGESTION

CATALOG = "retail_demo"
RAW_SCHEMA = f"{CATALOG}.raw"
BASE_PATH = "/Volumes/retail_demo/raw/retail_files/retail_delta_project"
BATCH_PATH = f"{BASE_PATH}/datasets/batch"
print("Source:", BATCH_PATH)
print("Target schema:", RAW_SCHEMA)

Source: /Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/batch
Target schema: retail_demo.raw


In [0]:
#Read Customers as raw strings

customers_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{BATCH_PATH}/customers_batch.csv")
)
display(customers_raw.limit(10))
customers_raw.printSchema()

customer_id,customer_name,city,segment,gender,signup_date,status
C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active
C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active
C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active
C00004,Customer_4,Bengaluru,Silver,Other,2025-09-08,active
C00005,Customer_5,Ahmedabad,Silver,Other,2025-10-27,inactive
C00006,Customer_6,Bengaluru,Platinum,Other,2024-10-11,active
C00007,Customer_7,Mumbai,Platinum,F,2024-10-11,active
C00008,Customer_8,Bengaluru,Gold,M,2024-04-04,inactive
C00009,Customer_9,Delhi,Gold,F,2025-09-10,active
C00010,Customer_10,Jaipur,Platinum,Other,2024-05-07,inactive


root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
from pyspark.sql.functions import current_timestamp, lit
customers_bronze = (
    customers_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("customers_batch.csv"))
    .withColumn("_source_type", lit("batch"))
)
display(customers_bronze.limit(10))

customer_id,customer_name,city,segment,gender,signup_date,status,_ingested_at,_source_file,_source_type
C00001,Customer_1,Delhi,Regular,Other,2024-10-08,active,2026-08-12T17:21:56.842Z,customers_batch.csv,batch
C00002,Customer_2,Bengaluru,Silver,Other,2024-04-14,active,2026-08-12T17:21:56.842Z,customers_batch.csv,batch
C00003,Customer_3,Gurugram,Platinum,M,2024-01-31,active,2026-08-12T17:21:56.842Z,customers_batch.csv,batch
C00004,Customer_4,Bengaluru,Silver,Other,2025-09-08,active,2026-08-12T17:21:56.842Z,customers_batch.csv,batch
C00005,Customer_5,Ahmedabad,Silver,Other,2025-10-27,inactive,2026-08-12T17:21:56.842Z,customers_batch.csv,batch
C00006,Customer_6,Bengaluru,Platinum,Other,2024-10-11,active,2026-08-12T17:21:56.842Z,customers_batch.csv,batch
C00007,Customer_7,Mumbai,Platinum,F,2024-10-11,active,2026-08-12T17:21:56.842Z,customers_batch.csv,batch
C00008,Customer_8,Bengaluru,Gold,M,2024-04-04,inactive,2026-08-12T17:21:56.842Z,customers_batch.csv,batch
C00009,Customer_9,Delhi,Gold,F,2025-09-10,active,2026-08-12T17:21:56.842Z,customers_batch.csv,batch
C00010,Customer_10,Jaipur,Platinum,Other,2024-05-07,inactive,2026-08-12T17:21:56.842Z,customers_batch.csv,batch


In [0]:
# Write customers to delta table
(
    customers_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retail_demo.raw.customers_bronze")
)

In [0]:
%sql
-- count number of rows 
SELECT COUNT(*) AS row_count
FROM retail_demo.raw.customers_bronze;

row_count
2560


In [0]:
products_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{BATCH_PATH}/products_batch.csv")
)
products_bronze = (
    products_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("products_batch.csv"))
    .withColumn("_source_type", lit("batch"))
)
display(products_bronze.limit(10))

product_id,product_name,category,brand,unit_price,status,created_date,_ingested_at,_source_file,_source_type
P00001,T-Shirt 1,Fashion,BrandC,unknown,discontinued,2025-02-15,2026-08-12T17:22:02.870Z,products_batch.csv,batch
P00002,Bedsheet 2,Home,BrandC,unknown,active,2024-04-20,2026-08-12T17:22:02.870Z,products_batch.csv,batch
P00003,Bedsheet 3,Home,BrandD,30753.26,active,2024-10-23,2026-08-12T17:22:02.870Z,products_batch.csv,batch
P00004,Oil 4,Grocery,BrandB,68176.44,active,2024-03-07,2026-08-12T17:22:02.870Z,products_batch.csv,batch
P00005,Oil 5,null,BrandC,51796.39,active,2024-12-12,2026-08-12T17:22:02.870Z,products_batch.csv,batch
P00006,Chair 6,Home,BrandC,10740.11,discontinued,2025-06-17,2026-08-12T17:22:02.870Z,products_batch.csv,batch
P00007,Jeans 7,Fashion,BrandC,26020.02,active,2024-04-19,2026-08-12T17:22:02.870Z,products_batch.csv,batch
P00008,Chair 8,Home,BrandB,8710.16,active,2023-03-23,2026-08-12T17:22:02.870Z,products_batch.csv,batch
P00009,Cream 9,Beauty,BrandA,71925.36,discontinued,2023-02-22,2026-08-12T17:22:02.870Z,products_batch.csv,batch
P00010,Chair 10,Home,BrandD,25422.83,discontinued,2025-12-02,2026-08-12T17:22:02.870Z,products_batch.csv,batch


In [0]:
(
    products_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retail_demo.raw.products_bronze")
)

In [0]:
%sql
-- count product rows
SELECT COUNT(*) AS row_count
FROM retail_demo.raw.products_bronze;

row_count
830


In [0]:
stores_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{BATCH_PATH}/stores_batch.csv")
)

stores_bronze = (
    stores_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("stores_batch.csv"))
    .withColumn("_source_type", lit("batch"))
)

display(stores_bronze.limit(10))
(
    stores_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retail_demo.raw.stores_bronze")
)

store_id,store_name,city,region,status,_ingested_at,_source_file,_source_type
S001,Store_1,Jaipur,Online,active,2026-08-12T17:22:08.607Z,stores_batch.csv,batch
S002,Store_2,Pune,West,active,2026-08-12T17:22:08.607Z,stores_batch.csv,batch
S003,Store_3,Ahmedabad,South,closed,2026-08-12T17:22:08.607Z,stores_batch.csv,batch
S004,Store_4,Ahmedabad,North,active,2026-08-12T17:22:08.607Z,stores_batch.csv,batch
S005,Store_5,Chennai,East,active,2026-08-12T17:22:08.607Z,stores_batch.csv,batch
S006,Store_6,Kolkata,East,closed,2026-08-12T17:22:08.607Z,stores_batch.csv,batch
S007,Store_7,Hyderabad,Online,active,2026-08-12T17:22:08.607Z,stores_batch.csv,batch
S008,Store_8,Chennai,North,closed,2026-08-12T17:22:08.607Z,stores_batch.csv,batch
S009,Store_9,Bengaluru,West,active,2026-08-12T17:22:08.607Z,stores_batch.csv,batch
S010,Store_10,Gurugram,East,active,2026-08-12T17:22:08.607Z,stores_batch.csv,batch


In [0]:
%sql
-- count stores 
SELECT COUNT(*) AS row_count
FROM retail_demo.raw.stores_bronze;

row_count
80


In [0]:
# Write orders to delta table

orders_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{BATCH_PATH}/orders_batch.csv")
)
orders_bronze = (
    orders_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("orders_batch.csv"))
    .withColumn("_source_type", lit("batch"))
)
display(orders_bronze.limit(10))
(
    orders_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retail_demo.raw.orders_bronze")
)

order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,_ingested_at,_source_file,_source_type
O0000001,2025-12-15 04:49:00,C01671,P00638,S003,4,30260.29,0.15,121041.16,CARD,cancelled,2026-08-12T17:22:13.798Z,orders_batch.csv,batch
O0000002,2026-03-13 03:29:00,C00713,P00768,S028,4,30049.26,0.1,102167.48,CARD,returned,2026-08-12T17:22:13.798Z,orders_batch.csv,batch
O0000003,2026-03-05 07:20:00,C02131,P00142,S072,5,59685.88,0.0,283507.93,COD,delivered,2026-08-12T17:22:13.798Z,orders_batch.csv,batch
O0000004,2025-12-28 13:56:00,C01399,P00031,S071,1,18705.7,0.05,15899.84,NETBANKING,returned,2026-08-12T17:22:13.798Z,orders_batch.csv,batch
O0000005,2025-12-31 17:43:00,C00383,P00564,S049,2,46727.14,0.15,84108.85,COD,cancelled,2026-08-12T17:22:13.798Z,orders_batch.csv,batch
O0000006,2026-03-11 02:59:00,C01745,P00101,S019,2,12387.77,0.05,21059.21,COD,returned,2026-08-12T17:22:13.798Z,orders_batch.csv,batch
O0000007,2025-12-26 03:16:00,C01633,P00555,S020,2,54612.57,0.05,98302.63,COD,returned,2026-08-12T17:22:13.798Z,orders_batch.csv,batch
O0000008,2025-12-09 20:57:00,C01960,P00771,S029,1,29725.2,0.0,28238.94,UPI,returned,2026-08-12T17:22:13.798Z,orders_batch.csv,batch
O0000009,2026-01-20 19:25:00,C00479,P00782,S050,4,84092.97,0.05,336371.88,UPI,delivered,2026-08-12T17:22:13.798Z,orders_batch.csv,batch
O0000010,2026-03-21 06:52:00,C00197,P00068,S049,5,31466.94,0.0,157334.7,COD,delivered,2026-08-12T17:22:13.798Z,orders_batch.csv,batch


In [0]:
%sql
-- count orders 
SELECT COUNT(*) AS row_count
FROM retail_demo.raw.orders_bronze;

row_count
12180


In [0]:
%sql
-- Final Bronze validation

SELECT 'customers_bronze' AS table_name,
       COUNT(*) AS row_count
FROM retail_demo.raw.customers_bronze
UNION ALL
SELECT 'orders_bronze',
       COUNT(*)
FROM retail_demo.raw.orders_bronze
UNION ALL
SELECT 'products_bronze',
       COUNT(*)
FROM retail_demo.raw.products_bronze
UNION ALL
SELECT 'stores_bronze',
       COUNT(*)
FROM retail_demo.raw.stores_bronze;

table_name,row_count
customers_bronze,2560
orders_bronze,12180
products_bronze,830
stores_bronze,80
